# Ring resonator optimization

In this notebook we optimize a ring resonator to place a resonance at a target wavelength,
using the same JAX gradient-based approach as the MZI optimization notebook.

By the end, you'll have a scaffold for the **multi-channel ring challenge**: optimize
multiple rings, each tuned to a different wavelength.

## Imports

In [ ]:
import gdsfactory as gf
import gdsfactoryplus as gfp
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import sax
from ipywidgets import interact

from gpdk.models.couplers import coupler_ring_coupling_area
from gpdk.models.waveguides import bend_euler as bend_euler_model

PDK = gfp.get_pdk()
gfp.register_cells()
jax.config.update("jax_enable_x64", True)

## Config

In [ ]:
wl = jnp.linspace(1.50, 1.60, 1001)  # [um] wavelength sweep

# Ring geometry defaults
radius = 10.0    # [um] bend radius
gap = 0.2        # [um] coupling gap
length_x0 = 4.0  # [um] racetrack coupling length
length_y = 0.6   # [um] vertical straight length

## Ring resonator layout

First, let's build the physical layout — this is the same `ring_single` component from the morning session.

In [ ]:
ring_layout = gf.components.ring_single(gap=gap, radius=radius, length_x=length_x0)
ring_layout.plot()

In [ ]:
scene = ring_layout.to_3d()
scene.show()

## Circuit model

To simulate and optimize the ring, we build an equivalent SAX circuit.
The all-pass ring consists of a bus-to-ring coupler with the ring cavity
looped back through straights and bends:

```
              st (length_x)
         bl ─────────────── br
         │                   │
    sl   │                   │   sr
 (len_y) │                   │ (len_y)
         │                   │
         └──── coupler ──────┘
               (cr)
          o1 ──────────── o2
```

The coupler model already includes the bend arcs at the bottom of the ring,
so we only need to add the top bends and straights.

The `coupler_ring` PDK model internally builds a sub-circuit each time
it's called. We pre-compile it once so that JAX can trace through it
during optimization.

In [ ]:
bend = gf.components.bend_euler(radius=radius, cross_section="strip")
bend_length = bend.info["length"]
print(f"Bend arc length: {bend_length:.2f} μm")

In [ ]:
_coupler_ring_circuit, _ = sax.circuit(
    netlist={
        "instances": {"bl": "bend_euler", "c": "coupler_ring", "br": "bend_euler"},
        "connections": {"bl,o2": "c,o2", "c,o3": "br,o1"},
        "ports": {"o1": "c,o1", "o2": "bl,o1", "o3": "br,o2", "o4": "c,o4"},
    },
    models={"coupler_ring": coupler_ring_coupling_area, "bend_euler": bend_euler_model},
)


def coupler_ring_jittable(*, wl=1.55, gap=0.2, radius=10.0, length_x=4.0, cross_section="strip", **kwargs):
    s = _coupler_ring_circuit(
        wl=wl,
        c={"length_x": length_x, "gap": gap, "radius": radius, "cross_section": cross_section},
        bl={"length": bend_length},
        br={"length": bend_length},
    )
    return sax.reciprocal(
        {("o1", "o4"): s["o1", "o4"], ("o1", "o3"): s["o1", "o3"],
         ("o2", "o4"): s["o2", "o4"], ("o2", "o3"): s["o2", "o3"]}
    )


ring_models = dict(PDK.models)
ring_models["coupler_ring"] = coupler_ring_jittable

In [ ]:
ring_netlist = {
    "instances": {
        "cr": "coupler_ring",
        "bl": "bend_euler",
        "br": "bend_euler",
        "sl": "straight",
        "sr": "straight",
        "st": "straight",
    },
    "connections": {
        "cr,o2": "sl,o1",
        "sl,o2": "bl,o2",
        "bl,o1": "st,o2",
        "st,o1": "br,o2",
        "br,o1": "sr,o1",
        "sr,o2": "cr,o3",
    },
    "ports": {
        "o1": "cr,o1",
        "o2": "cr,o4",
    },
}

gfp.show(ring_netlist)

In [ ]:
_ring, _ = sax.circuit(ring_netlist, ring_models)


@jax.jit
def ring_model(wl=1.55, length_x=4.0, gap=gap, radius=radius, length_y=length_y):
    """Evaluate the ring circuit with variable racetrack coupling length."""
    return _ring(
        wl=wl,
        cr={"length_x": length_x, "gap": gap, "radius": radius},
        st={"length": length_x},
        sl={"length": length_y},
        sr={"length": length_y},
        bl={"length": bend_length},
        br={"length": bend_length},
    )

## Interactive exploration

Use the slider to see how `length_x` shifts the ring resonances.
Increasing `length_x` extends the ring circumference, which reduces the
free spectral range and shifts the resonance positions.

In [ ]:
@interact(length_x=(0.5, 20.0, 0.5))
def show(length_x=length_x0):
    plt.figure(figsize=(10, 4))
    S = sax.sdict(ring_model(wl=wl, length_x=length_x))
    plt.plot(wl * 1e3, jnp.abs(S["o1", "o2"]) ** 2, color="C0")
    plt.xlabel("λ [nm]")
    plt.ylabel("Through-port power")
    plt.title(f"Ring resonator — length_x = {length_x:.1f} μm")
    plt.ylim(-0.05, 1.05)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

## Optimization

We want a resonance at exactly **1550.0 nm**. The loss function minimizes
through-port power at the target wavelength — when a resonance sits
there, the light couples into the ring and the through-port drops to a minimum.

Same approach as the MZI notebook: JAX autodiff gives us the gradient,
and Adam finds the optimal `length_x`.

In [ ]:
import jax.example_libraries.optimizers as opt
from tqdm.notebook import trange

wl_target = 1.5500  # [um]


@jax.jit
def loss_fn(length_x):
    S = ring_model(wl=wl_target, length_x=length_x)
    return jnp.mean(jnp.abs(S["o1", "o2"]) ** 2)

In [ ]:
grad_fn = jax.jit(jax.grad(loss_fn, argnums=0))

In [ ]:
initial_length_x = 4.0
init_fn, update_fn, params_fn = opt.adam(step_size=0.01)
state = init_fn(initial_length_x)

In [ ]:
def step_fn(step, state):
    """Runs one step of the optimizer."""
    params = params_fn(state)
    loss = loss_fn(params)
    grad = grad_fn(params)
    state = update_fn(step, grad, state)
    return loss, state

In [ ]:
for step in (pb := trange(300)):
    loss, state = step_fn(step, state)
    pb.set_postfix(loss=f"{loss:.6f}")

In [ ]:
optimized_length_x = float(params_fn(state))
print(f"Optimized length_x: {optimized_length_x:.3f} μm")
print(f"Target resonance:   {wl_target * 1e3:.1f} nm")

## Results

In [ ]:
S = sax.sdict(ring_model(wl=wl, length_x=optimized_length_x))

plt.figure(figsize=(10, 4))
plt.plot(wl * 1e3, jnp.abs(S["o1", "o2"]) ** 2)
plt.axvline(wl_target * 1e3, ls=":", color="black", label=f"Target: {wl_target*1e3:.1f} nm")
plt.xlabel("λ [nm]")
plt.ylabel("Through-port power")
plt.title("Optimized ring resonator")
plt.ylim(-0.05, 1.05)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
optimized_ring = gf.components.ring_single(
    gap=gap, radius=radius, length_x=optimized_length_x
)
optimized_ring.plot()

---

## Challenge: multi-channel ring bank

Optimize a bank of ring resonators, each tuned to a different channel wavelength.
This is the core building block of a WDM system — in a full design you'd use
add-drop rings (`ring_double`) to route each channel to a separate output,
but the optimization approach is identical.

### Tiers

| Level | Task | Hint |
|-------|------|------|
| **Bronze** | Optimize **one** ring for 1549.2 nm | Copy the optimization above, change `wl_target` |
| **Silver** | Optimize **4** rings for the channel wavelengths below | Loop over targets, optimize each independently |
| **Gold** | Ask the **AI agent** to design the full ring bank | Describe the spec in natural language and iterate |

In [ ]:
channel_wavelengths = jnp.array([1549.2, 1550.0, 1550.8, 1551.6]) / 1e3  # [um]

print("Target channel wavelengths:")
for i, wl_ch in enumerate(channel_wavelengths):
    print(f"  Channel {i + 1}: {wl_ch * 1e3:.1f} nm")

### Scoring

Run `score_wdm` with your optimized `length_x` values to see how well
each channel's resonance lines up. The score rewards both wavelength
accuracy and extinction ratio.

In [ ]:
def score_wdm(optimized_lengths):
    """Score a WDM filter design. Higher is better."""
    wl_min = float(channel_wavelengths.min()) - 0.005
    wl_max = float(channel_wavelengths.max()) + 0.005
    wl_fine = jnp.linspace(wl_min, wl_max, 5000)
    total = 0.0

    fig, axes = plt.subplots(1, len(optimized_lengths), figsize=(4 * len(optimized_lengths), 3), sharey=True)
    if len(optimized_lengths) == 1:
        axes = [axes]

    for i, (target, lx) in enumerate(zip(channel_wavelengths, optimized_lengths, strict=True)):
        S = sax.sdict(ring_model(wl=wl_fine, length_x=lx))
        T = jnp.abs(S["o1", "o2"]) ** 2

        mask = jnp.abs(wl_fine - target) < 0.002
        T_masked = jnp.where(mask, T, 1.0)
        min_idx = jnp.argmin(T_masked)
        resonance_wl = float(wl_fine[min_idx])
        T_min = float(T[min_idx])

        if T_min > 0.9:
            print(f"  Ch {i + 1}: no resonance found near {target*1e3:.1f} nm — check your length_x")
            channel_score = 0.0
        else:
            er = float(max(-10 * jnp.log10(T_min + 1e-10), 0))
            error_nm = abs(resonance_wl - float(target)) * 1e3
            channel_score = er / (1 + 10 * error_nm)
            print(
                f"  Ch {i + 1}: target={target*1e3:.1f} nm, resonance={resonance_wl*1e3:.2f} nm, "
                f"error={error_nm:.2f} nm, ER={er:.1f} dB, score={channel_score:.2f}"
            )

        total += channel_score

        axes[i].plot(wl_fine * 1e3, T)
        axes[i].axvline(float(target) * 1e3, ls=":", color="red")
        axes[i].set_title(f"Ch {i + 1}: {target*1e3:.1f} nm")
        axes[i].set_xlabel("λ [nm]")
        axes[i].set_ylim(-0.05, 1.05)
        axes[i].grid(True)

    axes[0].set_ylabel("Through-port power")
    plt.tight_layout()
    plt.show()

    print(f"\n  Total score: {total:.2f}")
    return total

### Your work starts here

**Bronze** — Optimize a single ring to hit one of the channel wavelengths.
Copy the optimization loop from above and change `wl_target`.

<details>
<summary>Hint 1 — What to copy</summary>

You need four pieces: `loss_fn`, `grad_fn`, the Adam optimizer setup, and the training loop.
Change `wl_target` to one of the channel wavelengths (e.g. `channel_wavelengths[0]`).

</details>

<details>
<summary>Hint 2 — Skeleton</summary>

```python
target = channel_wavelengths[0]

@jax.jit
def my_loss(length_x):
    S = ring_model(wl=target, length_x=length_x)
    return jnp.mean(jnp.abs(S["o1", "o2"]) ** 2)

my_grad = jax.jit(jax.grad(my_loss))
init_fn, update_fn, params_fn = opt.adam(step_size=0.01)
state = init_fn(4.0)

for step in trange(300):
    # ... same as above
```

</details>

In [ ]:
# Bronze: optimize one ring
# your code here


**Silver** — Optimize all 4 channels. Loop over `channel_wavelengths` and
collect the optimized `length_x` for each.

<details>
<summary>Hint 1 — Approach</summary>

Each channel is independent — optimize one ring at a time with a different
target wavelength. Collect the results in a list.

The tricky part: `loss_fn` and `grad_fn` are JIT-compiled, so they capture the
target wavelength at compile time. You need to re-define them inside the loop
for each new target, or pass the target as an argument.

</details>

<details>
<summary>Hint 2 — Skeleton</summary>

```python
optimized_lengths = []

for target in channel_wavelengths:
    @jax.jit
    def ch_loss(length_x, wl_t=target):
        S = ring_model(wl=wl_t, length_x=length_x)
        return jnp.mean(jnp.abs(S["o1", "o2"]) ** 2)

    ch_grad = jax.jit(jax.grad(ch_loss))
    init_fn, update_fn, params_fn = opt.adam(step_size=0.01)
    state = init_fn(4.0)

    for step in range(300):
        p = params_fn(state)
        state = update_fn(step, ch_grad(p), state)

    optimized_lengths.append(float(params_fn(state)))
    print(f"  {target*1e3:.1f} nm → length_x = {optimized_lengths[-1]:.3f} μm")
```

</details>

In [ ]:
# Silver: optimize 4 rings
# optimized_lengths = []
# for target in channel_wavelengths:
#     ...
#     optimized_lengths.append(result)


**Gold** — Use the AI agent to design the filter. Describe what you want
and let it write the optimization code for you.

<details>
<summary>Hint — Suggested prompt</summary>

Try something like:

> "Design a 4-channel WDM filter using ring resonators.
> Target wavelengths: 1549.2, 1550.0, 1550.8, 1551.6 nm.
> Optimize `ring_single` `length_x` for each channel using `ring_model`
> and JAX gradient optimization with Adam (step_size=0.01, 300 steps).
> Then call `score_wdm(optimized_lengths)` to score the result."

The agent has access to all the functions defined in this notebook.

</details>

In [ ]:
# Gold: AI-assisted design
# Open the AI agent and describe your WDM filter spec:
#   "Design a 4-channel WDM filter using ring resonators.
#    Target wavelengths: 1549.2, 1550.0, 1550.8, 1551.6 nm.
#    Optimize ring_single length_x for each channel.
#    Use the ring_model function and JAX gradient optimization."


### Score your result

In [ ]:
# Uncomment and fill in your optimized lengths:
# my_lengths = [length_ch1, length_ch2, length_ch3, length_ch4]
# score_wdm(my_lengths)

---

## Bonus: explore further

Finished the challenge? Here are some ideas to keep going.

### 1. Optimize gap and radius too

The optimization above only varies `length_x`. What if you also optimize
`gap` (coupling strength) or `radius` (bend loss vs. footprint)?

```python
# Multi-parameter optimization — pack into a dict
@jax.jit
def loss_multi(params):
    S = _ring(
        wl=wl_target,
        cr={"length_x": params["length_x"], "gap": params["gap"], "radius": radius},
        st={"length": params["length_x"]},
        sl={"length": length_y}, sr={"length": length_y},
        bl={"length": bend_length}, br={"length": bend_length},
    )
    return jnp.mean(jnp.abs(S["o1", "o2"]) ** 2)
```

### 2. Build an MZI lattice filter

The PDK has `mzi_lattice` — a cascade of MZI stages that can create
flat-top filter passbands. Try building and simulating one:

```python
lattice = gf.components.mzi_lattice(delta_lengths=[10.0, 20.0])
lattice.plot()
```

### 3. Explore the transceiver sample

See a full chip-scale design that uses ring modulators:

```python
from gpdk.samples.transceiver_rings import transceiver_rings
chip = transceiver_rings()
chip.plot()
```

### 4. Compare routing strategies

Run the A* routing and playdough differentiable routing samples
side by side. Check path length matching:

```python
%run ../gpdk/samples/sample_routing_astar.py
%run ../gpdk/samples/sample_routing_playdough_obstacles.py
```